# 8.7 High-Dimensional Data Visualisation

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 8: Dimensionality Reduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-08-dimensionality-reduction/08.07_high-dimensional-data-visualisation.ipynb)

Companion code for Section 8.7 of the book. Run the cells in order from top to bottom.

**Example 8.7.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, load_breast_cancer, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist
import warnings; warnings.filterwarnings('ignore')

# Load datasets
iris   = load_iris()
bc     = load_breast_cancer()
digits = load_digits()

X_iris = iris.data
y_iris = iris.target
names_iris = iris.feature_names
target_iris = iris.target_names

X_bc  = StandardScaler().fit_transform(bc.data)
y_bc  = bc.target

# ── Technique 1: Pair Plot ────────────────────────────────────────
iris_df = pd.DataFrame(X_iris, columns=names_iris)
iris_df['Species'] = [target_iris[y] for y in y_iris]

g = sns.pairplot(iris_df, hue='Species', diag_kind='kde',
                  plot_kws={'alpha':0.7, 's':30},
                  palette={'setosa':'#E53935','versicolor':'#1976D2','virginica':'#43A047'})
g.fig.suptitle('Pair Plot - All Pairwise Feature Relationships (Iris)',
              y=1.02, fontsize=14, fontweight='bold')
g.savefig('pairplot_iris.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Technique 2: Correlation Heatmap ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Correlation Analysis', fontsize=14, fontweight='bold')

# Iris correlation
ax = axes[0]
corr = pd.DataFrame(X_iris, columns=names_iris).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax, mask=mask, annot=True, fmt='.3f',
           cmap='RdYlGn', center=0, linewidths=0.5, vmin=-1, vmax=1,
           cbar_kws={'shrink':0.8})
ax.set_title('Iris Feature Correlations (4 features)')

# Breast cancer top features
ax = axes[1]
bc_df = pd.DataFrame(bc.data[:,:10], columns=bc.feature_names[:10])
corr_bc = bc_df.corr()
sns.heatmap(corr_bc, ax=ax, annot=True, fmt='.2f', cmap='RdYlGn',
           center=0, linewidths=0.3, cbar_kws={'shrink':0.8},
           annot_kws={'size':8})
ax.set_title('Breast Cancer Top-10 Feature Correlations')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)

plt.tight_layout()
plt.savefig('correlation_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Technique 3: Parallel Coordinates ────────────────────────────
from pandas.plotting import parallel_coordinates
fig, ax = plt.subplots(figsize=(12, 6))
iris_df_norm = iris_df.copy()
for col in names_iris:
    iris_df_norm[col] = (iris_df[col] - iris_df[col].min()) / (iris_df[col].max() - iris_df[col].min())
parallel_coordinates(iris_df_norm, 'Species', ax=ax,
                     color=['#E53935','#1976D2','#43A047'], alpha=0.4)
ax.set_title('Parallel Coordinates Plot - Iris\n(each line = one observation)',
            fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('parallel_coordinates.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Technique 4: RadViz ───────────────────────────────────────────
from pandas.plotting import radviz
fig, ax = plt.subplots(figsize=(8, 8))
radviz(iris_df, 'Species', ax=ax,
       color=['#E53935','#1976D2','#43A047'], alpha=0.7)
ax.set_title('RadViz - Radial Visualisation (Iris)',
            fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('radviz.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Technique 5: 3D PCA Scatter ──────────────────────────────────
fig = plt.figure(figsize=(12, 10))
ax  = fig.add_subplot(111, projection='3d')
pca3 = PCA(n_components=3)
Z3   = pca3.fit_transform(StandardScaler().fit_transform(X_bc))
colors = ['#E53935' if c==0 else '#1976D2' for c in y_bc]
ax.scatter(Z3[:,0], Z3[:,1], Z3[:,2], c=colors, s=20, alpha=0.7)
ax.set_xlabel(f'PC1 ({pca3.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca3.explained_variance_ratio_[1]:.1%})')
ax.set_zlabel(f'PC3 ({pca3.explained_variance_ratio_[2]:.1%})')
ax.set_title('3D PCA - Breast Cancer\n(Red=Malignant, Blue=Benign)',
            fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('pca_3d.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Technique 6: Feature distribution by class ────────────────────
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Feature Distributions by Class - Breast Cancer (Top 12 Features)',
            fontsize=14, fontweight='bold')

top_features_idx = np.argsort(np.abs(np.corrcoef(bc.data.T, y_bc)[:-1,-1]))[::-1][:12]

for idx, feat_idx in enumerate(top_features_idx):
    ax = axes[idx//4, idx%4]
    feat_name = bc.feature_names[feat_idx]
    feat_data = bc.data[:, feat_idx]

    for cls, color, label in [(0,'#E53935','Malignant'),(1,'#1976D2','Benign')]:
        sns.kdeplot(feat_data[y_bc==cls], ax=ax, color=color,
                   label=label, fill=True, alpha=0.4)

    ax.set_title(feat_name, fontsize=9, fontweight='bold')
    ax.set_xlabel('')
    if idx==0: ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Technique 7: Digit image reconstruction quality ───────────────
fig, axes = plt.subplots(3, 8, figsize=(16, 6))
fig.suptitle('PCA Reconstruction Quality - Handwritten Digits',
            fontsize=14, fontweight='bold')

sample_idx = np.arange(8)
X_d = digits.data[sample_idx]

# Original images
for i, idx in enumerate(sample_idx):
    axes[0, i].imshow(X_d[i].reshape(8,8), cmap='gray_r', interpolation='nearest')
    axes[0, i].set_title(f'Digit: {digits.target[idx]}', fontsize=10)
    axes[0, i].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=11, rotation=0, ha='right', labelpad=60)

# k=10 components reconstruction
for row, k in enumerate([10, 40], start=1):
    pca_d = PCA(n_components=k).fit(digits.data)
    Z_d   = pca_d.transform(X_d)
    X_rec = pca_d.inverse_transform(Z_d)
    var   = pca_d.explained_variance_ratio_.sum()
    for i in range(8):
        axes[row, i].imshow(X_rec[i].reshape(8,8), cmap='gray_r', interpolation='nearest')
        axes[row, i].axis('off')
    axes[row, 0].set_ylabel(f'k={k}\n({var:.0%} var)', fontsize=10,
                            rotation=0, ha='right', labelpad=60)

plt.tight_layout()
plt.savefig('digit_reconstruction.png', dpi=150, bbox_inches='tight')
plt.show()

print('All visualisation figures saved.')

**Example 8.7.2**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

# Complete pipeline: load → scale → reduce → cluster → classify
digits = load_digits()
X, y   = digits.data, digits.target

print('='*70)
print('COMPLETE DIMENSIONALITY REDUCTION PIPELINE - DIGITS (64 features)')
print('='*70)

# ── Step 1: Baseline (raw features) ──────────────────────────────
X_sc = StandardScaler().fit_transform(X)
lr   = LogisticRegression(max_iter=2000, random_state=42)
score_raw = cross_val_score(lr, X_sc, y, cv=5, scoring='accuracy').mean()
km = KMeans(n_clusters=10, n_init=10, random_state=42).fit(X_sc)
ari_raw = adjusted_rand_score(y, km.labels_)
sil_raw = silhouette_score(X_sc, km.labels_)
print(f'\n[1] Raw (64 features):')
print(f'    Classification accuracy: {score_raw:.4f}')
print(f'    Clustering ARI:          {ari_raw:.4f}')
print(f'    Silhouette Score:        {sil_raw:.4f}')

# ── Step 2: PCA preprocessing ─────────────────────────────────────
for n_pca in [10, 20, 40]:
    pca = PCA(n_components=n_pca)
    X_pca = pca.fit_transform(X_sc)
    var = pca.explained_variance_ratio_.sum()

    pipe = Pipeline([('lr', LogisticRegression(max_iter=2000, random_state=42))])
    score_pca = cross_val_score(pipe, X_pca, y, cv=5, scoring='accuracy').mean()

    km_pca = KMeans(n_clusters=10, n_init=10, random_state=42).fit(X_pca)
    ari_pca = adjusted_rand_score(y, km_pca.labels_)

    print(f'\n[2] PCA({n_pca}, {var:.0%} variance):')
    print(f'    Classification accuracy: {score_pca:.4f}')
    print(f'    Clustering ARI:          {ari_pca:.4f}')

# ── Step 3: t-SNE for visualisation ──────────────────────────────
X_pca50 = PCA(n_components=50).fit_transform(X_sc)
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
Z_tsne = tsne.fit_transform(X_pca50)

score_tsne = cross_val_score(
    LogisticRegression(max_iter=2000, random_state=42),
    Z_tsne, y, cv=5, scoring='accuracy').mean()
km_tsne = KMeans(n_clusters=10, n_init=10, random_state=42).fit(Z_tsne)
ari_tsne = adjusted_rand_score(y, km_tsne.labels_)
sil_tsne = silhouette_score(Z_tsne, km_tsne.labels_)

print(f'\n[3] PCA(50) → t-SNE(2):')
print(f'    Classification accuracy: {score_tsne:.4f}  ← not recommended for classification!')
print(f'    Clustering ARI:          {ari_tsne:.4f}')
print(f'    Silhouette Score:        {sil_tsne:.4f}')

# ── Final Comprehensive Visualisation ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Dimensionality Reduction - Complete Analysis (Digits Dataset)',
            fontsize=14, fontweight='bold')
palette_10 = plt.cm.tab10(np.linspace(0,1,10))

# 1. PCA 2D scatter
ax = axes[0,0]
Z_pca2 = PCA(n_components=2).fit_transform(X_sc)
for k in range(10):
    ax.scatter(Z_pca2[y==k,0], Z_pca2[y==k,1], s=8, alpha=0.7, color=palette_10[k], label=str(k))
ax.set_title(f'PCA (2D, {PCA(2).fit(X_sc).explained_variance_ratio_.sum():.0%} variance)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.grid(alpha=0.3)

# 2. PCA 3D scatter
ax = fig.add_subplot(2, 3, 2, projection='3d')
Z_pca3 = PCA(n_components=3).fit_transform(X_sc)
for k in range(10):
    ax.scatter(Z_pca3[y==k,0], Z_pca3[y==k,1], Z_pca3[y==k,2],
              s=5, alpha=0.6, color=palette_10[k])
ax.set_title('PCA (3D)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_zlabel('PC3')

# 3. t-SNE scatter
ax = axes[0,2]
for k in range(10):
    ax.scatter(Z_tsne[y==k,0], Z_tsne[y==k,1], s=8, alpha=0.8, color=palette_10[k], label=str(k))
    cx, cy = Z_tsne[y==k,0].mean(), Z_tsne[y==k,1].mean()
    ax.annotate(str(k), (cx,cy), fontsize=12, fontweight='bold', ha='center',
               bbox=dict(boxstyle='round,pad=0.2',fc='white',alpha=0.7))
ax.set_title('t-SNE (2D, perplexity=30)'); ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.legend(title='Digit', fontsize=7, ncol=2, markerscale=2); ax.grid(alpha=0.2)

# 4. Scree plot
ax = axes[1,0]
pca_full = PCA().fit(X_sc)
ax.bar(range(1,21), pca_full.explained_variance_ratio_[:20], color='#1976D2', alpha=0.8)
ax.plot(range(1,21), pca_full.explained_variance_ratio_[:20], 'ro-', ms=6)
ax2 = ax.twinx()
ax2.plot(range(1,65), pca_full.explained_variance_ratio_.cumsum(), 'g-', lw=2,
        label='Cumulative')
ax.set_xlabel('PC Index'); ax.set_ylabel('Variance Explained')
ax2.set_ylabel('Cumulative Variance', color='green')
ax.set_title('Scree Plot - Digits (64 features)'); ax.grid(alpha=0.3)

# 5. Method comparison
ax = axes[1,1]
methods  = ['Raw\n(64d)', 'PCA10\n(84%)', 'PCA20\n(93%)', 'PCA40\n(99%)', 't-SNE\n(2d)']
accuracies=[score_raw,
           cross_val_score(lr, PCA(10).fit_transform(X_sc), y, cv=5).mean(),
           cross_val_score(lr, PCA(20).fit_transform(X_sc), y, cv=5).mean(),
           cross_val_score(lr, PCA(40).fit_transform(X_sc), y, cv=5).mean(),
           score_tsne]
colors_bar = ['#43A047','#1976D2','#1976D2','#1976D2','#FF6F00']
bars = ax.bar(methods, accuracies, color=colors_bar, edgecolor='white')
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
           f'{acc:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('CV Accuracy (5-fold)'); ax.set_title('Classification Accuracy by Method')
ax.set_ylim(0.85, 1.01); ax.grid(alpha=0.3, axis='y')

# 6. Example digit images
ax = axes[1,2]
ax.axis('off')
# Show 10 example digits (one per class)
for k in range(10):
    idx = np.where(y==k)[0][0]
    ax_inset = ax.inset_axes([0.1*(k%5)+0.05, 0.45*(k//5)+0.05, 0.15, 0.40])
    ax_inset.imshow(X[idx].reshape(8,8), cmap='gray_r', interpolation='nearest')
    ax_inset.set_title(str(k), fontsize=10, pad=1)
    ax_inset.axis('off')
ax.set_title('Sample Digit Images (8×8 pixels = 64 features)')

plt.tight_layout()
plt.savefig('complete_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Pipeline Summary ===')
print('1. Scale features (StandardScaler) - always required')
print('2. PCA(50) for noise reduction and speed - good preprocessing')
print('3. PCA(2D) or t-SNE(2D) for visualisation - see cluster structure')
print('4. PCA(20) for classification - 93% variance, near-baseline accuracy')
print('5. Use raw or PCA features for ML - NEVER t-SNE features!')